# Plan-and-Solve

- 两阶段工作流：
    - Planning Phase：规划阶段。agent接收用户的完整问题，将问题分解，并制定出一个清晰、分步骤的行动计划。该阶段主要专注制定计划；
    - Solving Phase：执行阶段。严格按照计划中的步骤，逐一执行。每一步执行可能使一次独立的LLM调用，或者是对上一步结果的加工处理，直到计划中的所有步骤都完成，得到最终答案。该阶段除了要专注执行解决每个子问题外，还要进行状态管理，保证包含原始问题、完整计划、历史步骤与结果、当前步骤信息。

---

- 抽象工作流程：首先，规划模型$\pi_{plan}$根据原始问题 q 生成一个包含 n 个步骤的计划 $P=(p_1,p_2,...,p_n)$ :
$$P=\pi_{plan}(q)$$
随后，在执行阶段，执行模型$\pi_{solve}$逐一完成计划中的步骤。对于第 i 个步骤，其解决方案 $s_i$ 的生成会同时依赖于原始问题 q、完整计划 P 以及之前所有步骤的执行结果 $(s_1,...,s_{i-1})$ :
$$s_i = \pi_{solve}(q, P, (s_1,...,s_{i-1}))$$
最终的答案就是最后一个步骤的执行结果$s_n$

In [2]:
# prompt template - 规划阶段

PLANNER_PROMPT_TEMPLATE = """
你是一个顶级的AI规划专家。你的任务是将用户提出的复杂问题分解成一个由多个简单步骤组成的行动计划。
请确保计划中的每个步骤都是一个独立的、可执行的子任务，并且严格按照逻辑顺序排列。
你的输出必须是一个Python列表，其中每个元素都是一个描述子任务的字符串。

问题: {question}

请严格按照以下格式输出你的计划,```python与```作为前后缀是必要的:
```python
["步骤1", "步骤2", "步骤3", ...]
```
"""

In [8]:
import sys
sys.path.append(r"C:\Users\61075\PycharmProjects\learning\hello_agent")
from _LLM import AgentsLLM
from typing import List, Dict
import ast
from dotenv import load_dotenv

load_dotenv(r"C:\Users\61075\PycharmProjects\learning\hello_agent\.env")

True

In [14]:
class Planner:
    def __init__(self, llm_client: AgentsLLM):
        self.llm_client = llm_client

    def plan(self, question: str) -> List[str]:
        """根据用户问题生成一个行动计划"""
        prompt = PLANNER_PROMPT_TEMPLATE.format(question=question)

        # 为了生成计划，构建一个简单的消息列表
        messages = [{"role": "user", "content": prompt}]

        print("--- 正在生成计划 ---")
        # 使用流式输出来获取完整计划
        response_text = self.llm_client.think(messages) or ""

        print(f"%%% 计划已生成：\n{response_text}")

        # 解析LLM输出的列表字符串
        try:
            # 找到```python和```之间的内容
            plan_str = response_text.split("```python")[1].split("```")[0].strip()
            # 使用ast.literal_eval来安全地执行字符串，将其转换为Python列表
            plan = ast.literal_eval(plan_str)
            return plan if isinstance(plan, list) else []
        except (ValueError, SyntaxError, IndexError) as e:
            print(f"!!! 解析计划时出错：{e}")
            print(f"原始响应：{response_text}")
            return []
        except Exception as e:
            print(f"!!! 解析计划时发生未知错误：{e}")
            return []

In [9]:
# prompt template - 执行阶段

EXECUTOR_PROMPT_TEMPLATE = """
你是一位顶级的AI执行专家。你的任务是严格按照给定的计划，一步步地解决问题。
你将收到原始问题、完整的计划、以及到目前为止已经完成的步骤和结果。
请你专注于解决“当前步骤”，并仅输出该步骤的最终答案，不要输出任何额外的解释或对话。

# 原始问题:
{question}

# 完整计划:
{plan}

# 历史步骤与结果:
{history}

# 当前步骤:
{current_step}

请仅输出针对“当前步骤”的回答:
"""

In [10]:
class Executor:
    def __init__(self, llm_client: AgentsLLM):
        self.llm_client = llm_client

    def execute(self, question: str, plan: List[str]) -> str:
        """根据提示，逐步执行并解决问题"""
        history = ""    # 用于存储历史步骤和结果的字符串

        print("\n--- 正在执行计划 ---")
        for i, step in enumerate(plan):
            print(f"\n-> 正在执行步骤 {i+1}/{len(plan)}：{step}")

            prompt = EXECUTOR_PROMPT_TEMPLATE.format(
                question=question,
                plan=plan,
                history=history if history else "无",    # 如果时第一步，则历史为空
                current_step=step
            )

            messages = [{"role": "user", "content": prompt}]
            response_text = self.llm_client.think(messages) or ""

            # 更新历史记录
            history += f"步骤{i+1}: {step}\n结果: {response_text}" or ""

            print(f"%%% 步骤 {i+1} 已完成，结果：{response_text}")

        # 循环结束，最后一步的响应就是最终答案
        final_answer = response_text
        return final_answer

In [15]:
class PlanAndSolveAgent:
    def __init__(self, llm_client: AgentsLLM):
        """初始化agent，创建规划器和执行器"""
        self.llm_client = llm_client
        self.planner = Planner(self.llm_client)
        self.executor = Executor(self.llm_client)

    def run(self, question: str):
        """运行agent的完整流程，先规划，后执行"""
        print(f"\n--- 开始处理问题 ---\n问题：{question}")

        # 1.调用规划器生成计划
        plan = self.planner.plan(question)

        # 检查计划是否生成成功
        if not plan:
            print("\n--- 任务终止 ---\nERROR: 无法生成有效的行动计划")
            return

        # 2.调用执行器执行计划
        final_answer = self.executor.execute(question, plan)

        print(f"\n--- 任务完成 ---\n最终答案：{final_answer}")

In [16]:
try:
    llm_client = AgentsLLM()
    agent = PlanAndSolveAgent(llm_client)
    question = "一个水果店周一卖出了15个苹果。周二卖出的苹果数量是周一的两倍。周三卖出的数量比周二少了5个。请问这三天总共卖出了多少个苹果？"
    agent.run(question)
except Exception as e:
    print(f"{e}")


--- 开始处理问题 ---
问题：一个水果店周一卖出了15个苹果。周二卖出的苹果数量是周一的两倍。周三卖出的数量比周二少了5个。请问这三天总共卖出了多少个苹果？
--- 正在生成计划 ---
%%% 正在调用 llama-3.3-70b-versatile 模型...
%%% 大语言模型响应成功：
```python
["计算周一销售的苹果数量", "计算周二销售的苹果数量，周二的销售量是周一的两倍", "计算周三销售的苹果数量，周三的销售量比周二少5个", "将周一、周二和周三的销售数量相加，得出总共销售的苹果数量"]
```
%%% 计划已生成：
```python
["计算周一销售的苹果数量", "计算周二销售的苹果数量，周二的销售量是周一的两倍", "计算周三销售的苹果数量，周三的销售量比周二少5个", "将周一、周二和周三的销售数量相加，得出总共销售的苹果数量"]
```

--- 正在执行计划 ---

-> 正在执行步骤 1/4：计算周一销售的苹果数量
%%% 正在调用 llama-3.3-70b-versatile 模型...
%%% 大语言模型响应成功：
15
%%% 步骤 1 已完成，结果：15

-> 正在执行步骤 2/4：计算周二销售的苹果数量，周二的销售量是周一的两倍
%%% 正在调用 llama-3.3-70b-versatile 模型...
%%% 大语言模型响应成功：
30
%%% 步骤 2 已完成，结果：30

-> 正在执行步骤 3/4：计算周三销售的苹果数量，周三的销售量比周二少5个
%%% 正在调用 llama-3.3-70b-versatile 模型...
%%% 大语言模型响应成功：
25
%%% 步骤 3 已完成，结果：25

-> 正在执行步骤 4/4：将周一、周二和周三的销售数量相加，得出总共销售的苹果数量
%%% 正在调用 llama-3.3-70b-versatile 模型...
%%% 大语言模型响应成功：
15 + 30 + 25 = 70
%%% 步骤 4 已完成，结果：15 + 30 + 25 = 70

--- 任务完成 ---
最终答案：15 + 30 + 25 = 70
